# Pieces 2, 3 and 4

| Piece | What | Chooses |
|---|---|---|
| **3** | balanced losses: CE / weighted / focal / dice | best loss |
| **2** | joint sentence + token head | best lambda |
| **4** | offline distillation from SemiSOLD | best distill weight |

Piece 3 runs first because it is independent. Piece 2 must run before Piece 4,
because SemiSOLD's teacher scores are **sentence level** and distillation has
nothing to attach to without the sentence head.

Everything is chosen on **validation**. Only the last cell scores test.

---
### Setup
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Push your code, then edit `REPO` in cell 3
3. Run top to bottom

**Save to Drive** (cell 5) so a disconnect does not lose your results.

Rough timings on a T4: Piece 3 ~1h, Piece 2 ~1.5h, Piece 4 ~2h, final ~1h.


## 1. GPU


In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'


## 2. Install


In [ ]:
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
print('ready')


## 3. Code


In [ ]:
REPO = 'https://github.com/hatheem-r/project_DNN.git'   # <-- EDIT

import os, shutil
if os.path.exists('/content/project'): shutil.rmtree('/content/project')
%cd /content
!git clone -q $REPO project
%cd /content/project
!ls src/ notebooks/


## 4. Safety tests — do not skip


In [ ]:
!python tests/test_metrics.py | tail -2
!python tests/test_subword_alignment.py | tail -2


## 5. Mount Drive so results survive a disconnect

Colab wipes the session when it ends or times out. Everything below writes
to Drive as well as the session disk.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/sold_results'
import os
os.makedirs(OUT, exist_ok=True)
print('saving to', OUT)


## 6. fastText vectors

About 460 MB. Needed unless you pass `--minimal` everywhere.


In [ ]:
!mkdir -p embeddings results artifacts
![ -f embeddings/cc.si.300.vec.gz ] || wget -q --show-progress -O embeddings/cc.si.300.vec.gz https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
!ls -lh embeddings/


## 7. Smoke test — 1 seed, 3 epochs

Confirms everything runs before you commit hours. The score will be poor.


In [ ]:
!python notebooks/09_pieces_234.py --piece3 --seeds 1 --epochs 3 --patience 2 2>&1 | tail -15


## 8. PIECE 3 — balanced losses

Four losses, 3 seeds each, on validation.

**Read precision and recall, not just F1.** After Piece 1 the model sits near
precision 0.74 / recall 0.68, close to balanced. These losses all push toward
recall, so there is much less headroom than at Phase 1's 0.75/0.50. A loss that
lifts recall while collapsing precision is a net loss.

The CRF is off here: it computes its own sequence likelihood and cannot take
per-class weights. Step 7b showed it is redundant now, so this is
evidence-backed rather than convenient.


In [ ]:
!python notebooks/09_pieces_234.py --piece3 2>&1 | tee $OUT/piece3.txt | tail -30
!cp results/results_pieces234.csv $OUT/ 2>/dev/null; print('saved')


## 9. PIECE 2 — joint sentence + token head

`loss = token_loss + lambda * sentence_loss`, sweeping lambda over 0, 0.1,
0.3, 0.5, 1.0. Lambda 0 is the Piece 1 model exactly, so it is the control.

**Set `--loss` to whatever won Piece 3.** If Piece 3 was null, leave it as
`cross_entropy` — the simplest option that is not worse.

Even if lambda gives no gain, you still need a lambda > 0 for Piece 4.


In [ ]:
LOSS = 'cross_entropy'   # <-- set to the Piece 3 winner

!python notebooks/09_pieces_234.py --piece2 --loss $LOSS 2>&1 | tee $OUT/piece2.txt | tail -25
!cp results/results_pieces234.csv $OUT/ 2>/dev/null; print('saved')


## 10. PIECE 4 — offline distillation from SemiSOLD

Downloads 145,000 extra tweets carrying scores from eleven classifiers the SOLD
authors ran in 2022. We filter by teacher disagreement at their own optimal
threshold of 0.1 (~8,500 tweets; they found 0.15 added noise and made results
*worse*), average the top teachers into one soft target per tweet, and train
the sentence head to match.

**No pretrained language model is loaded, run, or backpropagated through.** The
scores are static columns in a public file. We consume a published artifact,
exactly as we consume the gold labels.

**Set `--lambda` to the Piece 2 winner.** If Piece 2 was null, use 0.3.


In [ ]:
LAM = 0.3   # <-- set to the Piece 2 winner

!python notebooks/09_pieces_234.py --piece4 --loss $LOSS --lambda $LAM 2>&1 | tee $OUT/piece4.txt | tail -30
!cp results/results_pieces234.csv $OUT/ 2>/dev/null; print('saved')


## 11. FINAL — winning combination, full train, test scored ONCE

Fill in the three winners below. Set `DW = 0` if distillation gave nothing.

This retrains the tokenizer, vocabulary and embedding matrix on all 7,500
tweets, then scores test once with 5 seeds.

**Whatever it gives is the result.** Do not re-run with different settings
because you dislike the number — that is tuning on test.


In [ ]:
LOSS = 'cross_entropy'   # <-- Piece 3 winner
LAM  = 0.3               # <-- Piece 2 winner (0 if you skip the sentence head)
DW   = 1.0               # <-- Piece 4 winner (0 to disable distillation)

!python notebooks/09_pieces_234.py --final --loss $LOSS --lambda $LAM --distill-weight $DW \
    2>&1 | tee $OUT/pieces234_final.txt | tail -30
!cp results/results_pieces234.csv $OUT/ 2>/dev/null; print('saved')


## 12. Confirm everything is on Drive


In [ ]:
!ls -la $OUT


---
## Expect nulls, and report them

Piece 1 already moved the model from precision 0.75 / recall 0.50 to roughly
0.74 / 0.68. All three remaining components push toward recall, so there is far
less left for them to fix than there was at Phase 1.

A careful null result is a legitimate finding, and three of them tell one
coherent story alongside the CRF result: **once the input representation is
fixed, the downstream corrections stop mattering.** That is a more interesting
paper than four components each contributing a little.

What would be dishonest is running these, seeing nothing, and quietly leaving
them out. Report every row.
